# 03 – Analyse

Korrelationsanalysen, Hypothesentests und tiefergehende Auswertungen
der Polar-Gesundheitsdaten (2014–2026).

## Inhalt
1. Setup & Daten laden
2. Ruhepuls-Entwicklung über die Zeit
3. Wochenmuster: Ruhepuls nach Wochentag (Heatmap)
4. Hypothese 1 – Training → Ruhepuls
5. Hypothese 2 – Schlaf → Ruhepuls
6. HRV-Entwicklung & Saisonalität
7. Trainingsvolumen & Sportarten
8. Tagesrhythmus: Herzfrequenz nach Stunde
9. Fazit & Key Insights

## 1 – Setup & Daten laden

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from db_loader import DatabricksLoader, secrets_pruefen

secrets_pruefen()

# ── Plot-Stil ────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
FARBEN = {
    'blau'   : '#4C72B0',
    'orange' : '#DD8452',
    'gruen'  : '#55A868',
    'rot'    : '#C44E52',
    'lila'   : '#8172B2',
    'tuerkis': '#64B5CD',
}
plt.rcParams.update({
    'figure.dpi'       : 120,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

# ── Daten laden ──────────────────────────────────────────────
db = DatabricksLoader()

df_act   = db.lade_activity()
df_train = db.lade_training()
df_hr    = db.lade_heartrate()
df_hrv   = db.lade_hrv()

for df in [df_act, df_train, df_hr, df_hrv]:
    if not df.empty:
        df['datum'] = pd.to_datetime(df['datum'])

# ── Gemeinsamer DataFrame: HR + Activity (nach Datum joinen) ─
df_joined = pd.DataFrame()
if not df_hr.empty and not df_act.empty:
    df_joined = pd.merge(
        df_hr[['datum','hr_ruhepuls','wochentag_nr','monat']],
        df_act[['datum','schritte','schlaf_stunden','met_minuten']],
        on='datum', how='inner'
    )
    print(f"Joined DataFrame: {len(df_joined)} Tage mit HR + Activity")

print("\n✅ Alle Daten geladen")

## 2 – Ruhepuls-Entwicklung über die Zeit

Langzeittrend des Ruhepulses von 2014 bis 2026 mit verschiedenen Glättungen.

In [ ]:
if not df_hr.empty:
    df_s = df_hr.sort_values('datum').copy()

    # Glättungen berechnen
    df_s['trend_6m']  = df_s['hr_ruhepuls'].rolling(180, min_periods=30, center=True).mean()
    df_s['trend_12m'] = df_s['hr_ruhepuls'].rolling(365, min_periods=60, center=True).mean()

    # Jahres-Perzentile für Konfidenzband
    df_s['p25'] = df_s['hr_ruhepuls'].rolling(180, min_periods=30, center=True).quantile(0.25)
    df_s['p75'] = df_s['hr_ruhepuls'].rolling(180, min_periods=30, center=True).quantile(0.75)

    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
    fig.suptitle('Ruhepuls-Entwicklung 2014–2026', fontsize=15)

    # Oben: Rohwerte + Glättungen
    ax = axes[0]
    ax.scatter(df_s['datum'], df_s['hr_ruhepuls'],
               s=1.5, alpha=0.25, color=FARBEN['blau'], label='Tageswert')
    ax.fill_between(df_s['datum'], df_s['p25'], df_s['p75'],
                    alpha=0.12, color=FARBEN['blau'], label='IQR (25–75%)')
    ax.plot(df_s['datum'], df_s['trend_6m'],
            color=FARBEN['orange'], lw=2, alpha=0.9, label='6-Monats-Glättung')
    ax.plot(df_s['datum'], df_s['trend_12m'],
            color=FARBEN['rot'], lw=2.5, ls='--', label='12-Monats-Glättung')
    ax.set_ylabel('Ruhepuls (bpm)')
    ax.legend(fontsize=9, loc='upper right')
    ax.set_title('Langzeittrend mit Glättung')

    # Unten: Jahres-Boxplots
    ax = axes[1]
    df_s['jahr'] = df_s['datum'].dt.year
    jahre = sorted(df_s['jahr'].unique())
    data_boxplot = [df_s[df_s['jahr'] == j]['hr_ruhepuls'].dropna().values
                    for j in jahre]
    bp = ax.boxplot(data_boxplot, labels=jahre, patch_artist=True,
                    medianprops={'color': FARBEN['rot'], 'lw': 2},
                    whiskerprops={'alpha': 0.5},
                    flierprops={'marker': '.', 'markersize': 3, 'alpha': 0.3})
    for patch in bp['boxes']:
        patch.set_facecolor(FARBEN['blau'])
        patch.set_alpha(0.4)
    ax.set_ylabel('Ruhepuls (bpm)')
    ax.set_xlabel('Jahr')
    ax.set_title('Jahresverteilungen (Boxplot)')
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

    # Jahres-Statistik
    print("\nJahres-Statistik Ruhepuls:")
    jahrstat = df_s.groupby('jahr')['hr_ruhepuls'].agg(
        ['mean','median','std','min','max','count']
    ).round(1)
    display(jahrstat)

## 3 – Wochenmuster: Ruhepuls-Heatmap nach Monat & Wochentag

In [ ]:
if not df_hr.empty:
    wochentage_de = ['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So']
    monate_de     = ['Jan','Feb','Mär','Apr','Mai','Jun',
                     'Jul','Aug','Sep','Okt','Nov','Dez']

    # Pivot: Monat (Zeile) × Wochentag (Spalte)
    pivot = df_hr.groupby(['monat','wochentag_nr'])['hr_ruhepuls'].mean().unstack()
    pivot.columns = [wochentage_de[i] for i in pivot.columns]
    pivot.index   = [monate_de[i-1] for i in pivot.index]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Ruhepuls-Muster: Monat × Wochentag', fontsize=14)

    # Heatmap
    ax = axes[0]
    sns.heatmap(
        pivot,
        ax=ax,
        cmap='RdYlGn_r',
        annot=True, fmt='.1f',
        linewidths=0.5,
        cbar_kws={'label': 'Ø Ruhepuls (bpm)'},
        vmin=pivot.values.min() - 1,
        vmax=pivot.values.max() + 1,
    )
    ax.set_title('Ø Ruhepuls nach Monat & Wochentag')
    ax.set_xlabel('Wochentag')
    ax.set_ylabel('Monat')

    # Monatsmittel (Saisonalität)
    ax = axes[1]
    monat_mean = df_hr.groupby('monat')['hr_ruhepuls'].agg(['mean','sem']).reset_index()
    ax.bar(
        range(1, 13), monat_mean['mean'],
        yerr=monat_mean['sem'] * 1.96,
        color=FARBEN['blau'], alpha=0.75, edgecolor='white',
        capsize=4, error_kw={'elinewidth': 1.2}
    )
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(monate_de)
    ax.set_title('Ø Ruhepuls nach Monat (95% CI)')
    ax.set_ylabel('Ruhepuls (bpm)')
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.show()

    # Interpretation
    sommer = monat_mean[monat_mean['monat'].isin([6,7,8])]['mean'].mean()
    winter = monat_mean[monat_mean['monat'].isin([12,1,2])]['mean'].mean()
    print(f"Ø Ruhepuls Sommer (Jun–Aug): {sommer:.1f} bpm")
    print(f"Ø Ruhepuls Winter (Dez–Feb): {winter:.1f} bpm")
    print(f"Differenz Sommer vs. Winter: {winter - sommer:+.1f} bpm")

## 4 – Hypothese 1: Training → Ruhepuls

**Hypothese:** Mehr Training korreliert mit einem tieferen Ruhepuls.

Methode: Wöchentliche Trainingsminuten vs. Ø Ruhepuls der Folgewoche.

In [ ]:
if not df_train.empty and not df_hr.empty:

    # Trainingsminuten pro Woche aggregieren
    df_train_w = (
        df_train
        .set_index('datum')
        .resample('W')['dauer_min']
        .sum()
        .reset_index()
        .rename(columns={'dauer_min': 'trainingsmin_woche'})
    )

    # Ruhepuls pro Woche
    df_hr_w = (
        df_hr
        .set_index('datum')
        .resample('W')['hr_ruhepuls']
        .mean()
        .reset_index()
    )

    # Merge: Trainingswoche → Ruhepuls der *gleichen* Woche
    df_hyp1 = pd.merge(df_train_w, df_hr_w, on='datum', how='inner').dropna()

    # Pearson-Korrelation
    r, p = stats.pearsonr(df_hyp1['trainingsmin_woche'], df_hyp1['hr_ruhepuls'])
    n = len(df_hyp1)

    # Spearman (robuster gegenüber Ausreissern)
    r_sp, p_sp = stats.spearmanr(df_hyp1['trainingsmin_woche'], df_hyp1['hr_ruhepuls'])

    print("═" * 55)
    print("  Hypothese 1: Trainingsminuten → Ruhepuls")
    print("═" * 55)
    print(f"  n Wochen          : {n}")
    print(f"  Pearson r         : {r:+.4f}")
    print(f"  Pearson p-Wert    : {p:.4f} {'✅ signifikant' if p < 0.05 else '❌ nicht signifikant'}")
    print(f"  Spearman r        : {r_sp:+.4f}")
    print(f"  Spearman p-Wert   : {p_sp:.4f}")
    print(f"  Erklärte Varianz  : {r**2 * 100:.2f}% (R²)")
    print("═" * 55)
    print(f"\n  Interpretation: r = {r:+.2f} → {'schwache' if abs(r) < 0.3 else 'moderate' if abs(r) < 0.6 else 'starke'} Korrelation")
    print(f"  Das Trainingsvolumen erklärt nur {r**2*100:.1f}% der Ruhepuls-Varianz.")
    print(f"  Confounding-Variablen (Stress, Schlaf, Gesundheit) dominieren.")

    # Scatterplot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Hypothese 1: Trainingsminuten → Ruhepuls', fontsize=13)

    # Scatter mit Regressionslinie
    ax = axes[0]
    ax.scatter(
        df_hyp1['trainingsmin_woche'], df_hyp1['hr_ruhepuls'],
        alpha=0.25, s=15, color=FARBEN['blau']
    )
    # Regressionsgerade
    m, b = np.polyfit(df_hyp1['trainingsmin_woche'], df_hyp1['hr_ruhepuls'], 1)
    x_line = np.linspace(df_hyp1['trainingsmin_woche'].min(),
                          df_hyp1['trainingsmin_woche'].max(), 100)
    ax.plot(x_line, m * x_line + b, color=FARBEN['rot'], lw=2,
            label=f'Regression (r={r:+.2f})')
    ax.set_xlabel('Trainingsminuten / Woche')
    ax.set_ylabel('Ø Ruhepuls (bpm)')
    ax.set_title('Streudiagramm')
    ax.legend(fontsize=9)

    # Zeitreihe: Trainingsvolumen vs. Ruhepuls
    ax = axes[1]
    ax2 = ax.twinx()
    # Monatliche Aggregate für Übersichtlichkeit
    df_train_m = df_train.set_index('datum').resample('ME')['dauer_min'].sum().reset_index()
    df_hr_m    = df_hr.set_index('datum').resample('ME')['hr_ruhepuls'].mean().reset_index()
    ax.bar(df_train_m['datum'], df_train_m['dauer_min'] / 60,
           width=20, alpha=0.35, color=FARBEN['gruen'], label='Trainingsstunden/Monat')
    ax2.plot(df_hr_m['datum'], df_hr_m['hr_ruhepuls'],
             color=FARBEN['rot'], lw=1.5, label='Ø Ruhepuls')
    ax.set_ylabel('Trainingsstunden / Monat', color=FARBEN['gruen'])
    ax2.set_ylabel('Ruhepuls (bpm)', color=FARBEN['rot'])
    ax.set_title('Zeitreihe: Training & Ruhepuls')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

    plt.tight_layout()
    plt.show()

## 5 – Hypothese 2: Schlaf → Ruhepuls

**Hypothese:** Längerer Schlaf korreliert mit einem tieferen Ruhepuls am Folgetag.

In [ ]:
if not df_joined.empty:

    # Schlaf heute → Ruhepuls morgen (Lag-1)
    df_lag = df_joined[['datum','schlaf_stunden','hr_ruhepuls']].copy()
    df_lag['hr_morgen'] = df_lag['hr_ruhepuls'].shift(-1)
    df_lag = df_lag.dropna()

    # Nur physiologisch sinnvolle Schlafdauern
    df_lag = df_lag[(df_lag['schlaf_stunden'] >= 3) &
                    (df_lag['schlaf_stunden'] <= 12)]

    r, p       = stats.pearsonr(df_lag['schlaf_stunden'], df_lag['hr_morgen'])
    r_sp, p_sp = stats.spearmanr(df_lag['schlaf_stunden'], df_lag['hr_morgen'])

    print("═" * 55)
    print("  Hypothese 2: Schlafdauer → Ruhepuls (Lag +1 Tag)")
    print("═" * 55)
    print(f"  n Tage            : {len(df_lag)}")
    print(f"  Pearson r         : {r:+.4f}")
    print(f"  Pearson p-Wert    : {p:.4f} {'✅ signifikant' if p < 0.05 else '❌ nicht signifikant'}")
    print(f"  Spearman r        : {r_sp:+.4f}")
    print(f"  Erklärte Varianz  : {r**2 * 100:.2f}% (R²)")
    print("═" * 55)
    print(f"\n  Interpretation: r = {r:+.2f} → kein relevanter linearer Zusammenhang.")
    print(f"  Ruhepuls wird kaum durch die vorherige Nacht bestimmt.")

    # Visualisierung
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Hypothese 2: Schlafdauer → Ruhepuls (Folgetag)', fontsize=13)

    # Scatter
    ax = axes[0]
    ax.scatter(df_lag['schlaf_stunden'], df_lag['hr_morgen'],
               alpha=0.2, s=12, color=FARBEN['lila'])
    m, b = np.polyfit(df_lag['schlaf_stunden'], df_lag['hr_morgen'], 1)
    x_line = np.linspace(df_lag['schlaf_stunden'].min(),
                          df_lag['schlaf_stunden'].max(), 100)
    ax.plot(x_line, m * x_line + b, color=FARBEN['rot'], lw=2,
            label=f'Regression (r={r:+.2f})')
    ax.set_xlabel('Schlafdauer (h)')
    ax.set_ylabel('Ruhepuls Folgetag (bpm)')
    ax.set_title('Streudiagramm')
    ax.legend(fontsize=9)

    # Ruhepuls nach Schlaf-Kategorie (Boxplot)
    ax = axes[1]
    df_lag['schlaf_kat'] = pd.cut(
        df_lag['schlaf_stunden'],
        bins=[0, 5, 6, 7, 8, 9, 99],
        labels=['<5h', '5–6h', '6–7h', '7–8h', '8–9h', '>9h']
    )
    schlaf_order = ['<5h', '5–6h', '6–7h', '7–8h', '8–9h', '>9h']
    sns.boxplot(
        data=df_lag, x='schlaf_kat', y='hr_morgen',
        order=schlaf_order,
        palette='Purples',
        width=0.5, ax=ax
    )
    ax.set_xlabel('Schlafdauer')
    ax.set_ylabel('Ruhepuls Folgetag (bpm)')
    ax.set_title('Ruhepuls nach Schlaf-Kategorie')

    plt.tight_layout()
    plt.show()

    # Mittelwerte pro Schlaf-Kategorie
    print("\nØ Ruhepuls nach Schlaf-Kategorie:")
    print(df_lag.groupby('schlaf_kat', observed=True)['hr_morgen']
          .agg(['mean','count']).round(2).to_string())

## 6 – HRV-Entwicklung & Saisonalität

In [ ]:
if not df_hrv.empty:
    df_hrv['monat'] = df_hrv['datum'].dt.month
    df_hrv['jahr']  = df_hrv['datum'].dt.year

    # Monatsmittel
    df_hrv_m = (
        df_hrv.set_index('datum')
        .resample('ME')
        .agg({'hrv_rmssd': 'mean', 'hrv_sdnn': 'mean'})
        .reset_index()
    )

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    fig.suptitle('HRV-Analyse: Entwicklung & Saisonalität', fontsize=14)

    # RMSSD Zeitreihe
    ax = axes[0, 0]
    ax.plot(df_hrv_m['datum'], df_hrv_m['hrv_rmssd'],
            color=FARBEN['lila'], lw=1.5, alpha=0.85)
    rmssd_trend = df_hrv_m['hrv_rmssd'].rolling(6, min_periods=2).mean()
    ax.plot(df_hrv_m['datum'], rmssd_trend,
            color=FARBEN['rot'], lw=2.5, ls='--', label='6-Monats-Glättung')
    ax.fill_between(df_hrv_m['datum'], df_hrv_m['hrv_rmssd'],
                    alpha=0.12, color=FARBEN['lila'])
    ax.set_title('RMSSD Monatsmittel')
    ax.set_ylabel('RMSSD (ms)')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

    # SDNN Zeitreihe
    ax = axes[0, 1]
    ax.plot(df_hrv_m['datum'], df_hrv_m['hrv_sdnn'],
            color=FARBEN['tuerkis'], lw=1.5, alpha=0.85)
    ax.fill_between(df_hrv_m['datum'], df_hrv_m['hrv_sdnn'],
                    alpha=0.12, color=FARBEN['tuerkis'])
    ax.set_title('SDNN Monatsmittel')
    ax.set_ylabel('SDNN (ms)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

    # Saisonalität RMSSD
    ax = axes[1, 0]
    monate_de = ['Jan','Feb','Mär','Apr','Mai','Jun',
                 'Jul','Aug','Sep','Okt','Nov','Dez']
    hrv_monat = df_hrv.groupby('monat')['hrv_rmssd'].agg(['mean','sem']).reset_index()
    ax.bar(hrv_monat['monat'], hrv_monat['mean'],
           yerr=hrv_monat['sem'] * 1.96,
           color=FARBEN['lila'], alpha=0.7, edgecolor='white',
           capsize=4)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(monate_de)
    ax.set_title('RMSSD nach Monat (Saisonalität, 95% CI)')
    ax.set_ylabel('RMSSD (ms)')

    # HRV vs. Ruhepuls Scatter
    ax = axes[1, 1]
    if not df_hr.empty:
        df_hrv_hr = pd.merge(
            df_hrv[['datum','hrv_rmssd']],
            df_hr[['datum','hr_ruhepuls']],
            on='datum', how='inner'
        ).dropna()
        r_hv, p_hv = stats.pearsonr(
            df_hrv_hr['hrv_rmssd'], df_hrv_hr['hr_ruhepuls']
        )
        ax.scatter(df_hrv_hr['hrv_rmssd'], df_hrv_hr['hr_ruhepuls'],
                   alpha=0.2, s=10, color=FARBEN['lila'])
        m2, b2 = np.polyfit(df_hrv_hr['hrv_rmssd'], df_hrv_hr['hr_ruhepuls'], 1)
        xl = np.linspace(df_hrv_hr['hrv_rmssd'].min(),
                          df_hrv_hr['hrv_rmssd'].max(), 100)
        ax.plot(xl, m2 * xl + b2, color=FARBEN['rot'], lw=2,
                label=f'r={r_hv:+.2f}, p={p_hv:.3f}')
        ax.set_xlabel('HRV RMSSD (ms)')
        ax.set_ylabel('Ruhepuls (bpm)')
        ax.set_title('HRV RMSSD vs. Ruhepuls')
        ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    # Sommer vs. Winter HRV
    sommer_rmssd = df_hrv[df_hrv['monat'].isin([6,7,8])]['hrv_rmssd'].mean()
    winter_rmssd = df_hrv[df_hrv['monat'].isin([12,1,2])]['hrv_rmssd'].mean()
    print(f"Ø RMSSD Sommer (Jun–Aug): {sommer_rmssd:.1f} ms")
    print(f"Ø RMSSD Winter (Dez–Feb): {winter_rmssd:.1f} ms")
    print(f"Saisonaler Unterschied  : {sommer_rmssd - winter_rmssd:+.1f} ms")

## 7 – Trainingsvolumen & Sportarten

In [ ]:
if not df_train.empty:
    df_train['jahr'] = df_train['datum'].dt.year

    # Top-5 Sportarten für Zeitreihe
    top5_sports = df_train['sport'].value_counts().head(5).index.tolist()

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    fig.suptitle('Trainingsanalyse', fontsize=14)

    # Stunden pro Jahr gestapelt nach Sportart
    ax = axes[0, 0]
    df_stunden = (
        df_train[df_train['sport'].isin(top5_sports)]
        .groupby(['jahr','sport'])['dauer_min']
        .sum() / 60
    ).unstack(fill_value=0)
    df_stunden.plot(
        kind='bar', stacked=True, ax=ax,
        colormap='tab10', edgecolor='white', width=0.75
    )
    ax.set_title('Trainingsstunden pro Jahr (Top-5 Sportarten)')
    ax.set_xlabel('Jahr')
    ax.set_ylabel('Stunden')
    ax.legend(fontsize=7, loc='upper left')
    plt.setp(ax.get_xticklabels(), rotation=45)

    # Laufen: Distanz pro Jahr
    ax = axes[0, 1]
    df_laufen = df_train[df_train['sport'] == 'RUNNING'].copy()
    if not df_laufen.empty:
        lauf_jahr = df_laufen.groupby('jahr').agg(
            distanz=('distanz_km', 'sum'),
            einheiten=('datum', 'count')
        ).reset_index()
        ax2 = ax.twinx()
        ax.bar(lauf_jahr['jahr'], lauf_jahr['distanz'],
               color=FARBEN['orange'], alpha=0.7, edgecolor='white',
               label='Gesamtdistanz (km)')
        ax2.plot(lauf_jahr['jahr'], lauf_jahr['einheiten'],
                 color=FARBEN['rot'], lw=2, marker='o', ms=5,
                 label='Anzahl Läufe')
        ax.set_title('Laufen: Distanz & Einheiten pro Jahr')
        ax.set_ylabel('Distanz (km)', color=FARBEN['orange'])
        ax2.set_ylabel('Anzahl Läufe', color=FARBEN['rot'])
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
    else:
        ax.text(0.5, 0.5, 'Keine Laufdaten', ha='center', va='center',
                transform=ax.transAxes)

    # Herzfrequenz-Zonen (wenn HR-Daten vorhanden)
    ax = axes[1, 0]
    df_hr_train = df_train[df_train['hr_avg'].notna() & df_train['hr_max'].notna()].copy()
    if not df_hr_train.empty:
        ax.scatter(df_hr_train['dauer_min'], df_hr_train['hr_avg'],
                   alpha=0.25, s=10, color=FARBEN['blau'],
                   label='Ø HR')
        ax.scatter(df_hr_train['dauer_min'], df_hr_train['hr_max'],
                   alpha=0.15, s=10, color=FARBEN['rot'],
                   label='Max HR')
        ax.set_title('Trainingsdauer vs. Herzfrequenz')
        ax.set_xlabel('Trainingsdauer (min)')
        ax.set_ylabel('Herzfrequenz (bpm)')
        ax.set_xlim(0, 300)
        ax.legend(fontsize=8)

    # Monatliche Trainingsfrequenz (Heatmap Jahr × Monat)
    ax = axes[1, 1]
    df_train['monat'] = df_train['datum'].dt.month
    heatmap_data = df_train.groupby(['jahr','monat']).size().unstack(fill_value=0)
    sns.heatmap(
        heatmap_data,
        ax=ax, cmap='YlOrRd',
        linewidths=0.3,
        xticklabels=['J','F','M','A','M','J','J','A','S','O','N','D'],
        cbar_kws={'label': 'Anzahl Trainings'}
    )
    ax.set_title('Trainingsfrequenz: Jahr × Monat')
    ax.set_xlabel('Monat')
    ax.set_ylabel('Jahr')

    plt.tight_layout()
    plt.show()

## 8 – Tagesrhythmus: Herzfrequenz nach Stunde

Aus den 24/7-Herzfrequenzdaten wird der durchschnittliche Tagesrhythmus rekonstruiert.

In [ ]:
# Tagesrhythmus direkt aus Databricks abfragen
# (Stunden-Aggregat aus den Rohdaten, falls verfügbar)
try:
    df_tagesrhy = db.abfrage(f"""
        SELECT
            HOUR(DATEADD(SECOND, seconds_from_day_start,
                 CAST(datum AS TIMESTAMP))) AS stunde,
            AVG(hr_value)                  AS hr_mean,
            COUNT(*)                       AS n
        FROM {db.catalog}.{db.schema}.heartrate_raw
        GROUP BY stunde
        ORDER BY stunde
    """)
except Exception:
    df_tagesrhy = pd.DataFrame()

# Fallback: Wochentag-Muster aus aggregierter HR-Tabelle simulieren
if df_tagesrhy.empty:
    print("Hinweis: Rohdaten-Tabelle 'heartrate_raw' nicht verfügbar.")
    print("Zeige stattdessen: Ruhepuls-Muster nach Wochentag & Monat.")

    if not df_hr.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        fig.suptitle('Herzfrequenz-Muster', fontsize=13)

        # Wochentag-Boxplot
        wochentage_de = ['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So']
        df_hr['wt_label'] = df_hr['wochentag_nr'].map(
            dict(enumerate(wochentage_de))
        )
        ax = axes[0]
        sns.violinplot(
            data=df_hr, x='wt_label', y='hr_ruhepuls',
            order=wochentage_de, palette='Blues',
            inner='quartile', ax=ax
        )
        ax.set_title('Ruhepuls-Verteilung nach Wochentag')
        ax.set_xlabel('Wochentag')
        ax.set_ylabel('Ruhepuls (bpm)')

        # Jahres-Entwicklung Ruhepuls
        ax = axes[1]
        df_hr['jahr'] = df_hr['datum'].dt.year
        jahresmittel = df_hr.groupby('jahr')['hr_ruhepuls'].mean()
        ax.plot(jahresmittel.index, jahresmittel.values,
                color=FARBEN['blau'], lw=2.5, marker='o', ms=7)
        ax.fill_between(jahresmittel.index, jahresmittel.values,
                        alpha=0.15, color=FARBEN['blau'])
        ax.set_title('Ø Ruhepuls pro Jahr')
        ax.set_xlabel('Jahr')
        ax.set_ylabel('Ruhepuls (bpm)')

        plt.tight_layout()
        plt.show()
else:
    # Tagesrhythmus-Plot falls Rohdaten vorhanden
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df_tagesrhy['stunde'], df_tagesrhy['hr_mean'],
            color=FARBEN['blau'], lw=2.5, marker='o', ms=5)
    ax.fill_between(df_tagesrhy['stunde'], df_tagesrhy['hr_mean'],
                    alpha=0.15, color=FARBEN['blau'])
    ax.set_xticks(range(0, 24))
    ax.set_xlabel('Stunde des Tages')
    ax.set_ylabel('Ø Herzfrequenz (bpm)')
    ax.set_title('Tagesrhythmus: Durchschnittliche Herzfrequenz nach Stunde')
    plt.tight_layout()
    plt.show()

## 9 – Fazit & Key Insights

In [ ]:
print("╔" + "═" * 57 + "╗")
print("║  KEY INSIGHTS – Polar Health Analytics               ║")
print("╠" + "═" * 57 + "╣")

if not df_hr.empty:
    df_hr['jahr'] = df_hr['datum'].dt.year
    früheste_jahr = df_hr['jahr'].min()
    späteste_jahr = df_hr['jahr'].max()
    hr_früh = df_hr[df_hr['jahr'] == früheste_jahr]['hr_ruhepuls'].mean()
    hr_spät = df_hr[df_hr['jahr'] == späteste_jahr]['hr_ruhepuls'].mean()
    print(f"║  Ruhepuls {früheste_jahr}: {hr_früh:.1f} bpm → {späteste_jahr}: {hr_spät:.1f} bpm "
          f"({hr_spät - hr_früh:+.1f})       ║")

if not df_hrv.empty:
    s_rv = df_hrv[df_hrv['datum'].dt.month.isin([6,7,8])]['hrv_rmssd'].mean()
    w_rv = df_hrv[df_hrv['datum'].dt.month.isin([12,1,2])]['hrv_rmssd'].mean()
    print(f"║  HRV RMSSD Sommer {s_rv:.1f}ms > Winter {w_rv:.1f}ms           ║")

print(f"║                                                       ║")
print(f"║  Hypothese 1 (Training → Ruhepuls): r ≈ -0.04        ║")
print(f"║  → Kein relevanter linearer Zusammenhang              ║")
print(f"║                                                       ║")
print(f"║  Hypothese 2 (Schlaf → Ruhepuls): r ≈ +0.02          ║")
print(f"║  → Kein relevanter linearer Zusammenhang              ║")
print(f"║                                                       ║")
print(f"║  Ruhepuls zeigt klare Saisonalität:                   ║")
print(f"║  Sommer tiefer als Winter (Hitze / Aktivität)         ║")
print("╚" + "═" * 57 + "╝")

db.schliessen()
print("\n✅ Analyse abgeschlossen – weiter mit 04_dashboard.ipynb")